# Vector Databases

In [1]:
import numpy as np
import time
from sentence_transformers import SentenceTransformer

c:\AAA\Personal\AIDE_Prep\CRag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer('models/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8516.94it/s]
BertModel LOAD REPORT from: models/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Corpus

documents = [
    # AI/ML
    "Neural networks learn patterns through backpropagation and gradient descent",
    "Random forests combine multiple decision trees for robust predictions",
    "Transfer learning fine-tunes pre-trained models on domain-specific data",
    "Batch normalization stabilizes training by normalizing layer inputs",
    "Dropout randomly deactivates neurons to prevent overfitting",
    # RAG/LLM
    "RAG retrieves relevant documents before generating a response",
    "Vector embeddings capture semantic meaning in dense numerical form",
    "Chunking splits large documents into smaller retrievable segments",
    "Re-ranking reorders initial retrieval results using cross-encoders",
    "Prompt engineering designs effective instructions for language models",
    # Infrastructure
    "Kubernetes manages containerized workloads with automatic scaling",
    "Docker images package application code with all dependencies",
    "CI/CD pipelines automate testing and deployment of code changes",
    "Load balancers distribute traffic across multiple server instances",
    "Message queues like Kafka decouple producers from consumers",
    # Data
    "Feature engineering creates informative variables from raw data",
    "Data pipelines extract transform and load data between systems",
    "Time series analysis detects patterns and trends in sequential data",
    "Anomaly detection identifies unusual patterns that differ from normal behavior",
    "A/B testing compares two variants to determine which performs better",
]

In [4]:
print('Encoding documents')
doc_embeddings = model.encode(documents)
print(f'Shape: {doc_embeddings.shape}')

Encoding documents
Shape: (20, 384)


# Section 1: FAISS - Facebook AI Similarity Search

In [5]:
import faiss

In [6]:
# Flat Index
print('\n--- 1a: FAISS Flat Index ---')
dimension = doc_embeddings.shape[1]

embeddings_f32 = doc_embeddings.astype(np.float32)

# IndexFlatIP = Inner Product (for cosine sim, normalize first!)
# IndexFlatL2 = Euclidean Distance

faiss.normalize_L2(embeddings_f32)
index_flat = faiss.IndexFlatIP(dimension)
index_flat.add(embeddings_f32)

print(f'Index type: Flat (exact search)')
print(f'Vectors stored: {index_flat.ntotal}')


--- 1a: FAISS Flat Index ---
Index type: Flat (exact search)
Vectors stored: 20


In [7]:
query = 'How do neural networks learn'
query_emb = model.encode([query]).astype(np.float32)
faiss.normalize_L2(query_emb)

k = 5
distances, indices = index_flat.search(query_emb, k)

print(f"\nQuery: '{query}'")
print(f"Top {k} results")

for i, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
    print(f" {i}. [{dist:.4f}] {documents[idx]}")


Query: 'How do neural networks learn'
Top 5 results
 1. [0.7376] Neural networks learn patterns through backpropagation and gradient descent
 2. [0.3951] Batch normalization stabilizes training by normalizing layer inputs
 3. [0.3692] Transfer learning fine-tunes pre-trained models on domain-specific data
 4. [0.3303] Dropout randomly deactivates neurons to prevent overfitting
 5. [0.2284] Vector embeddings capture semantic meaning in dense numerical form


In [11]:
# FIASS IVF Index (Approaximate Nearest Neighbor)
# IVF = Inverted File Index

# Concept: Documents ko CLUSTERS mein divide karo.
# Query time pe sirf nearby clusters search karo, sab nahi.
#
# Soch aise: Agar tujhe Ahmedabad mein restaurant dhundhna hai,
# tu poore India mein nahi dhundhega — pehle Gujarat filter karega,
# phir Ahmedabad, phir nearby area. Same concept.
 
nlist = 4  # number of clusters (production mein 100-1000+)
quantizer = faiss.IndexFlatIP(dimension)
index_ivf = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT)

# IVF needs training
index_ivf.train(embeddings_f32)
index_ivf.add(embeddings_f32)

In [ ]:
# nprobe = kitne clusters search karne hain (default 1)
# Zyada nprobe = better recall, but slower. Kam = fast, but kuch results miss ho sakte hain.

index_ivf.nprobe = 1  # sirf 1 nearest cluster search karo

distances, indices = index_ivf.search(query_emb, k)

print(f"\n[IVF search, nprobe={index_ivf.nprobe}]")
print(f"Query: '{query}'")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
    if idx == -1:      # IVF -1 return karta hai agar cluster mein itne results nahi mile
        continue
    print(f" {i}. [{dist:.4f}] {documents[idx]}")

In [ ]:
# nprobe badhao -> recall improve hoti hai. nprobe == nlist matlab poora search (= Flat jaisa exact)
index_ivf.nprobe = nlist   # saare clusters search karo
distances, indices = index_ivf.search(query_emb, k)

print(f"\n[IVF search, nprobe={index_ivf.nprobe} (= all clusters, ab exact match Flat ke barabar)]")
print(f"Query: '{query}'")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
    if idx == -1:
        continue
    print(f" {i}. [{dist:.4f}] {documents[idx]}")

print("\nTradeoff: nprobe=1 -> fast, recall kam | nprobe=nlist -> slow, recall full")

# Section 2: FAISS HNSW - Hierarchical Navigable Small World

In [ ]:
# HNSW = graph-based ANN. Vectors ko ek multi-layer graph mein connect karta hai.
# Search ek "skip-list" jaisa hai: upar wali layer mein lambi jump, neeche fine search.
# Production mein sabse popular (Qdrant, Weaviate, pgvector sab isi pe based hain).
#
# Soch: pehle highway pakdo (top layer, fast long jumps),
# phir city roads (mid), phir gali-mohalla (bottom, exact neighbor).

M = 16  # har node ke kitne neighbors (graph connectivity). Zyada M = better recall, zyada memory.
index_hnsw = faiss.IndexHNSWFlat(dimension, M, faiss.METRIC_INNER_PRODUCT)

# efConstruction = build quality (zyada = better graph, slow build)
index_hnsw.hnsw.efConstruction = 40
index_hnsw.add(embeddings_f32)   # HNSW ko training nahi chahiye (IVF ke ulta)

# efSearch = query-time exploration depth (zyada = better recall, slow search)
index_hnsw.hnsw.efSearch = 32

distances, indices = index_hnsw.search(query_emb, k)
print(f"\n[HNSW search, M={M}, efSearch={index_hnsw.hnsw.efSearch}]")
print(f"Query: '{query}'")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
    print(f" {i}. [{dist:.4f}] {documents[idx]}")

# Section 3: Index Comparison - Flat vs IVF vs HNSW

In [ ]:
# Recall@k = ANN index ne Flat (ground truth) ke kitne results match kiye.
# Speed measure karne ke liye ek hi query baar-baar chalate hain.

def recall_at_k(approx_idx, truth_idx):
    return len(set(approx_idx) & set(truth_idx)) / len(truth_idx)

# Ground truth = exact Flat search
flat_dist, flat_idx = index_flat.search(query_emb, k)
truth = flat_idx[0]

index_ivf.nprobe = 1
indexes = {
    "Flat (exact)": index_flat,
    f"IVF (nprobe=1)": index_ivf,
    f"HNSW (efSearch={index_hnsw.hnsw.efSearch})": index_hnsw,
}

print(f"{'Index':<28}{'Recall@'+str(k):<12}{'Avg query (ms)':<16}")
print("-" * 56)
for name, idx in indexes.items():
    # warm-up + timing over 200 runs
    for _ in range(5):
        idx.search(query_emb, k)
    t0 = time.perf_counter()
    for _ in range(200):
        d, ii = idx.search(query_emb, k)
    elapsed_ms = (time.perf_counter() - t0) / 200 * 1000
    rec = recall_at_k(ii[0], truth)
    print(f"{name:<28}{rec:<12.2f}{elapsed_ms:<16.4f}")

print("\nTakeaway:")
print(" - Flat: 100% recall, slow on large data (har vector se compare).")
print(" - IVF:  fast, recall nprobe pe depend karti hai.")
print(" - HNSW: fast + high recall, par zyada memory + build time.")
print(" Chhote dataset (<10k) pe Flat hi best. Millions of vectors pe IVF/HNSW.")

# Section 4: Persistence + Metadata (real RAG ke liye)

In [ ]:
import os, json

# FAISS sirf vectors store karta hai, text/metadata NAHI.
# Isliye index ko disk pe save karo + ek alag mapping (id -> text/source) rakho.

os.makedirs("faiss_store", exist_ok=True)

# 1. Index save
faiss.write_index(index_flat, "faiss_store/docs.index")

# 2. Metadata save (id -> text + jo bhi extra info chahiye)
metadata = {i: {"text": doc, "doc_id": i} for i, doc in enumerate(documents)}
with open("faiss_store/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print(f"  index    -> faiss_store/docs.index ({os.path.getsize('faiss_store/docs.index')} bytes)")
print(f"  metadata -> faiss_store/metadata.json")

In [ ]:
# Load back (jaise ek naye session/process mein karoge)
loaded_index = faiss.read_index("faiss_store/docs.index")
with open("faiss_store/metadata.json") as f:
    loaded_meta = json.load(f)   # JSON keys string ban jaati hain -> str(idx) use karo

print(f"Loaded index: {loaded_index.ntotal} vectors\n")

# Ek clean search() function jo score + metadata dono lautaye
def search(query_text, k=3):
    q = model.encode([query_text]).astype(np.float32)
    faiss.normalize_L2(q)
    scores, ids = loaded_index.search(q, k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        results.append({
            "score": round(float(score), 4),
            "text": loaded_meta[str(idx)]["text"],
        })
    return results

q = "container orchestration and scaling"
print(f"Query: '{q}'")
for i, r in enumerate(search(q, k=3), 1):
    print(f"  {i}. [{r['score']}] {r['text']}")